Propósito: Filtra los tickers propuestos para traerlos filtrados por condiciones.

Usa parametros del indicador RSI de 60min, 1d y 7d.

Este filtro busca capturar movimientos de la siguiente forma: Activo sobrevendido en 1d y 60m; la temporalidad baja indica el movimiento de reversion alcista en conjunto con una temporalidad diaria subiendo. La temporalidad más larga indica la tendencia del activo en largo plazo.

In [17]:
%pip install -q pandas-ta yfinance lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
# Importa las librerias requeridas.
import pandas as pd
import requests
import io
import yfinance as yf
import pandas_ta as ta

In [19]:
#Scrapea Wikipedia para acceder a los nombres de los tickers del S&P500.

#1.1 Define la URL y un 'User-Agent' para evitar el bloqueo.
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
header = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}

#1.2 Realiza la petición con el header de seguridad
r = requests.get(url, headers=header)

#1.3 Lee la tabla desde el texto de la respuesta, usando io.StringIO para forzar la interpretación como string HTML
tables = pd.read_html(io.StringIO(r.text))
df_spy = tables[0]

#1.4 Extrae la lista de Tickers
tickers_a_analizar = [t.replace('.', '-') for t in df_spy['Symbol'].tolist()]

#1.5 Añade tickers de interes.

mis_tickers = ['^VIX', 'SPY', 'SPXL' , 'TQQQ' , 'QQQ' , 'URA' , 'XLI' , 'XLF' , 'XLRE' , 'XLB' , 'XLY' , 'XLE', 'XLV' ,
                'XBI' , 'XLC' , 'BTC-USD' , 'DIA']

tickers_a_analizar.extend(mis_tickers)

print(f"Se han cargado {len(tickers_a_analizar)} tickers...")

Se han cargado 520 tickers...


In [20]:
#DESCARGA DE DATOS TEMPORALIDAD 60 MINUTOS.

tickers_limpios = tickers_a_analizar

print(f"Descargando datos para {len(tickers_limpios)} activos...")
data_60m = yf.download(tickers_limpios, period="2mo", interval="60m", group_by='ticker', progress=False)

Descargando datos para 520 activos...


In [29]:
#ANALISIS TEMPORALIDAD 60MIN.

empresas_60min = []

for ticker in tickers_limpios:
    try:
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables para validación.
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]

        # Lógica de Filtrado 60min
        if (rsi_act <= 35 
            and rsi_act >= rsi_ant
                ):
            
            empresas_60min.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                #Añade el MACD.
                'MACD': round(macd_act, 3)
                
            })
    except Exception as e:
      print(f"Error en {ticker}: {e}")

print("\n" + "="*95)
if empresas_60min:
    df_final = pd.DataFrame(empresas_60min)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales detectadas.")


 Ticker  RSI actual  RSI anterior     MACD
  GOOGL        26.1          24.6   -2.780
   TWLO        26.2          23.6   -9.846
   GOOG        26.3          24.2   -2.756
   META        27.9          22.5   -8.533
   SPXL        28.5          27.9   -3.241
    SPY        29.6          29.0   -2.724
    PLD        30.3          29.3   -1.454
   HOOD        31.5          28.6   -1.965
   SCHW        31.7          29.3   -1.126
    CMG        32.0          28.9   -0.190
   SNPS        32.2          29.2  -10.296
   SKYD        32.2          31.9   -0.379
    CEG        32.6          27.1   -4.429
     BG        32.7          32.4   -0.220
BTC-USD        32.9          29.6 -166.602
   AMZN        33.1          29.7   -1.908
    XLC        34.9          31.2   -0.298


In [22]:
# DESCARGA DATOS DIARIOS
print(f"Descargando datos DIARIOS para {len(tickers_limpios)} activos...")
data_1d = yf.download(tickers_limpios, period="2mo", interval="1d", group_by='ticker', progress=False)


Descargando datos DIARIOS para 520 activos...


In [30]:
# ANÁLISIS DIARIO

empresas_diario = []

for ticker in tickers_limpios:
    try:
        df = data_1d[ticker].dropna()
        if len(df) < 30: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=14, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_ant = df['RSI_14'].iloc[-2]
        rsi_act = df['RSI_14'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        

        precio_act = df['Close'].iloc[-1]

        # LÓGICA DE FILTRADO DIARIO
        estuvo_sobrevendido = rsi_act <= 35 or rsi_ant <= 35
        dejo_de_caer = rsi_act >= rsi_ant

        if estuvo_sobrevendido:
            
            empresas_diario.append({
                'Ticker': ticker,
                #'Prec diario': round(precio_act, 2),
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                'MACD': round(macd_act, 3),
                #'MACD_SIGNAL': round(sig_act , 3)
            })
    except:
        continue

print("\n" + "="*95)


print("PAPELES DETECTADOS EN DIARIO:")
if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales diarias.")


PAPELES DETECTADOS EN DIARIO:
Ticker  RSI actual  RSI anterior    MACD
   MGM        16.2          18.3  -2.787
   STT        21.8          23.7  -4.620
   WDC        22.0          25.0 -26.768
   STX        22.6          25.6 -44.595
   VRT        22.6          24.8  -9.829
    MS        23.5          24.9  -6.626
  SNDK        23.7          26.9 -64.447
   ALB        23.9          26.6  -7.165
   BAC        24.4          25.8  -2.105
   ARE        25.6          27.2  -2.117
   MOS        25.6          26.2  -1.329
  AMAT        25.8          29.1  -8.334
  FLEX        25.8          30.0  -2.805
   CAT        26.2          26.8 -16.664
   FIX        26.3          28.2 -26.973
  LRCX        26.4          29.8  -6.374
  KLAC        26.7          31.0  -1.870
  AKAM        26.8          29.5  -4.243
  SKYD        27.1          29.0  -0.740
   APO        27.2          28.7  -4.063
   URA        27.4          28.9  -2.053
   UAL        27.8          32.9  -3.010
  SMCI        27.9        

In [31]:
#TICKERS REPETIDOS EN AMBAS TEMPORALIDADES.

tickers_60 = [e['Ticker'] for e in empresas_60min]
ticker_1d = [e['Ticker'] for e in empresas_diario]

tickers_comunes = list(set(tickers_60) & set(ticker_1d)) # Interseccion de los elementos en la lista.

print(tickers_comunes)


['CMG', 'SCHW', 'PLD', 'SKYD', 'HOOD']


In [25]:
# DESCARGA DATOS SEMANAL.
print(f"Descargando datos SEMANALES para {len(tickers_limpios)} activos...")
data_7d = yf.download(tickers_limpios, period="1y", interval="1wk", group_by='ticker', progress=False)

Descargando datos SEMANALES para 520 activos...


In [32]:
# ANÁLISIS SEMANAL

empresas_semanal = []

for ticker in tickers_limpios:
    try:
        df = data_7d[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=10, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_act = df['RSI_10'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]

        # Lógica de filtrado.
        if rsi_act <= 44 and (rsi_act > rsi_ant):
        
            empresas_semanal.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                'MACD': round(macd_act, 3),
            })
    except:
        continue

print("\n" + "="*95)
print("ACTIVOS DETECTADOS EN SEMANAL.")
if empresas_semanal:
    df_final = pd.DataFrame(empresas_semanal)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales semanales.")


ACTIVOS DETECTADOS EN SEMANAL.
Ticker  RSI actual  RSI anterior     MACD
  WYNN        22.5          21.3   -7.923
   PNR        24.3          20.6   -8.638
   LII        28.3          27.8  -40.476
   CRH        29.7          23.9   -7.450
   FRT        30.9          30.3   -0.578
   LVS        31.3          21.9   -4.127
  CHTR        31.9          25.8  -16.968
   ROL        32.5          24.5   -5.408
  APTV        32.6          28.6   -6.324
   CMI        32.8          29.0  -19.488
    WY        32.8          28.1   -1.097
   MOS        33.2          32.8   -0.790
   KEY        33.6          33.3   -0.156
   PNC        33.8          33.6    1.093
  LULU        33.8          30.1  -13.990
   MAA        34.5          30.9   -3.302
   MCD        34.6          24.1  -13.535
   TXT        34.6          29.6   -3.723
  PCAR        35.0          27.7   -1.274
   BXP        35.0          34.3   -0.538
  FICO        35.3          25.8 -148.038
   XYL        35.4          31.5   -5.371
  